In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.columns.tolist())
df.head()

(891, 12)
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [37]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
print(df.isna().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64


In [38]:
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.")
print(df["Title"].value_counts())

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64


In [39]:
df["FamilySize"] = df ["SibSp"] + df ["Parch"] + 1
df ["IsAlone"] = (df["FamilySize"] == 1).astype(int)

print(df[["SibSp", 'Parch', 'FamilySize', "IsAlone"]].head(10))

   SibSp  Parch  FamilySize  IsAlone
0      1      0           2        0
1      1      0           2        0
2      0      0           1        1
3      1      0           2        0
4      0      0           1        1
5      0      0           1        1
6      0      0           1        1
7      3      1           5        0
8      0      2           3        0
9      1      0           2        0


In [40]:
columns_to_keep = ['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked','Title', 'FamilySize', 'IsAlone']
df = df[columns_to_keep]
df.head()

,Survived,Pclass,Sex,Age,Fare,Embarked,Title,FamilySize,IsAlone
0,0,3,male,22.0,7.2500,S,Mr,2,0
1,1,1,female,38.0,71.2833,C,Mrs,2,0
2,1,3,female,26.0,7.9250,S,Miss,1,1
3,1,1,female,35.0,53.1000,S,Mrs,2,0
4,0,3,male,35.0,8.0500,S,Mr,1,1


In [41]:
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=False)

print(df.columns.tolist())
print()
df.head()

['Survived', 'Pclass', 'Age', 'Fare', 'FamilySize', 'IsAlone', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Capt', 'Title_Col', 'Title_Countess', 'Title_Don', 'Title_Dr', 'Title_Jonkheer', 'Title_Lady', 'Title_Major', 'Title_Master', 'Title_Miss', 'Title_Mlle', 'Title_Mme', 'Title_Mr', 'Title_Mrs', 'Title_Ms', 'Title_Rev', 'Title_Sir']



,Survived,Pclass,Age,Fare,FamilySize,IsAlone,Sex_female,Sex_male,Embarked_C,Embarked_Q,...,Title_Major,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir
0,0,3,22.0,7.2500,2,0,False,True,False,False,...,False,False,False,False,False,True,False,False,False,False
1,1,1,38.0,71.2833,2,0,True,False,True,False,...,False,False,False,False,False,False,True,False,False,False
2,1,3,26.0,7.9250,1,1,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,1,1,35.0,53.1000,2,0,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,0,3,35.0,8.0500,1,1,False,True,False,False,...,False,False,False,False,False,True,False,False,False,False


In [42]:
X=df.drop(columns=['Survived'])
y=df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print ('Train:', X_train.shape, 'Test:', X_test.shape)
print('Доля выживших в train:', round(y_train.mean(), 4))
print('Доля выживших в test:', round(y_test.mean(), 4))


Train: (712, 27) Test: (179, 27)
Доля выживших в train: 0.3834
Доля выживших в test: 0.3855


In [43]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [44]:
print('пропуски в xtrain:', X_train.isna().sum().sum())
print('пропуски в xtest:', X_test.isna().sum().sum())
print()
print('где именно')
print(X_train.isna().sum()[X_train.isna().sum()>0])

пропуски в xtrain: 0
пропуски в xtest: 0

где именно
Series([], dtype: int64)


In [45]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred=model.predict(X_test_scaled)
acc=accuracy_score(y_test, y_pred)
print(f'Accuracy (Day 5): {acc:.2%}')

Accuracy (Day 5): 83.80%


In [48]:
baseline=0.8
print(f'baseline (day 4): {baseline}')
print(f'Day 5: {acc}')
print(f'Прирост: {(acc - baseline)}')

baseline (day 4): 0.8
Day 5: 0.8379888268156425
Прирост: 0.03798882681564242


In [51]:
confficlients = pd.Series(model.coef_[0], index=X.columns)
print(confficlients.sort_values(ascending=False))

Sex_female        0.507293
Title_Master      0.457716
Title_Mrs         0.243984
Title_Major       0.195561
Title_Sir         0.182689
Title_Mlle        0.150239
Title_Ms          0.147475
Fare              0.145503
Title_Lady        0.127301
Embarked_Q        0.112979
Embarked_C        0.041184
Title_Col         0.019120
Title_Mme         0.000000
Title_Capt        0.000000
Title_Countess    0.000000
Title_Dr         -0.002386
Title_Miss       -0.004734
Embarked_S       -0.104413
IsAlone          -0.169043
Title_Jonkheer   -0.179517
Title_Don        -0.191345
Title_Rev        -0.339048
Title_Mr         -0.343689
Age              -0.353390
Sex_male         -0.507293
FamilySize       -0.715664
Pclass           -0.844526
dtype: float64


In [56]:
from sklearn.tree import DecisionTreeClassifier

tree=DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

y_predt=tree.predict(X_test_scaled)
acct=accuracy_score(y_test, y_predt)
print(f'Accuracy (Day 5): {acct:.2%}')

imp = pd.Series(tree.feature_importances_, index=X.columns)
print(imp.sort_values(ascending=False).head(10))

Accuracy (Day 5): 78.21%
Title_Mr      0.533333
Fare          0.163582
Pclass        0.131667
FamilySize    0.073424
Title_Rev     0.041899
Age           0.035906
Title_Dr      0.011171
Title_Don     0.009018
IsAlone       0.000000
Sex_female    0.000000
dtype: float64


C:\Users\sirot\mlfirst+\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
